# Silver Layer — HCPCS Events (`hcpcsevents`)

**Source -> Target:** `<source_schema>.hcpcsevents` -> `<target_schema>.hcpcsevents` (defaults `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`). The target is replaced via `CREATE OR REPLACE`; downstream Gold depends on it.

**Dependency:** the dictionary-coverage profiling cell reads `<target_schema>.d_hcpcs` (run `06_d_hcpcs` first). It is informational only and is skipped if that table does not exist.

**Objective:** Standardize patient-level HCPCS events while preserving patient identifiers, hospital admissions, sequence numbers, HCPCS codes and source descriptions.

**Grain:** One source HCPCS event record. Per MIMIC-IV `hosp.hcpcsevents`, `subject_id`, `hadm_id`, `hcpcs_cd` and `seq_num` are NOT NULL; `hcpcs_cd` links to `d_hcpcs.code`. Repeated codes or sequence numbers will be investigated rather than automatically removed.

**Silver rules:** Preserve every Bronze record and its clinical meaning, standardize data types and whitespace, validate data quality, and reconcile Bronze–Silver counts. No deletion, deduplication, filtering, imputation or aggregation.

**Duplicate handling:** Exact duplicate rows (all columns identical) are counted and reported in one aggregation pass, but they never block publication and are never removed. The official documentation defines no unique key for this table, so identical rows may be legitimate repeated source records. No row-level duplicate flag is added: duplicates are not defined by the documentation and no profiling evidence supports a flag definition. Profile the reported counts and revisit only if evidence shows a specific, documented duplicate pattern.

**Validation policy:** Row preservation, the ordered schema and the documented NOT NULL columns (`subject_id`, `hadm_id`, `hcpcs_cd`, `seq_num`) are asserted unconditionally. Blank `hcpcs_cd` strings are not addressed by the documentation and are reported only.

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
# Configure Bronze source and Silver target schemas (widgets allow standalone and dbutils.notebook.run use)
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TABLE = "hcpcsevents"
SOURCE_TABLE = f"{BRONZE}.{TABLE}"
TARGET_TABLE = f"{SILVER}.{TABLE}"

# Expected published contract (ordered column -> type)
EXPECTED_SCHEMA = {
    "subject_id": "bigint",
    "hadm_id": "bigint",
    "chartdate": "date",
    "hcpcs_cd": "string",
    "seq_num": "int",
    "short_description": "string",
}

In [0]:
# Load and inspect patient HCPCS event records
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

print("Bronze rows:", bronze_df.count())

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- chartdate: date (nullable = true)
 |-- hcpcs_cd: string (nullable = true)
 |-- seq_num: integer (nullable = true)
 |-- short_description: string (nullable = true)



subject_id,hadm_id,chartdate,hcpcs_cd,seq_num,short_description
10000068,25022803,2160-03-04,99218,1,Hospital observation services
10000084,29888819,2160-12-28,G0378,1,Hospital observation per hr
10000108,27250926,2163-09-27,99219,1,Hospital observation services
10000117,22927623,2181-11-15,43239,1,Digestive system
10000117,22927623,2181-11-15,G0378,2,Hospital observation per hr
10000161,22148160,2163-08-20,99219,1,Hospital observation services
10000248,20600184,2192-11-30,99219,1,Hospital observation services
10000280,25852320,2151-03-18,99219,1,Hospital observation services
10000635,20642640,2143-12-23,G0378,1,Hospital observation per hr
10000635,26134563,2136-06-19,G0378,1,Hospital observation per hr


Bronze rows: 186074


In [0]:
# Profile source records and missing values
bronze_profile = bronze_df.select(
    F.count("*").alias("total_rows"),
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in bronze_df.columns
    ]
)

display(bronze_profile)

total_rows,null_subject_id,null_hadm_id,null_chartdate,null_hcpcs_cd,null_seq_num,null_short_description
186074,0,0,0,0,0,0


In [0]:
# Standardize HCPCS event identifiers, dates, codes and descriptions
silver_df = bronze_df.select(
    F.col("subject_id").cast("long").alias("subject_id"),
    F.col("hadm_id").cast("long").alias("hadm_id"),
    F.col("chartdate").cast("date").alias("chartdate"),
    F.trim(F.col("hcpcs_cd").cast("string")).alias("hcpcs_cd"),
    F.col("seq_num").cast("int").alias("seq_num"),
    F.trim(F.col("short_description").cast("string")).alias("short_description")
)

silver_df.printSchema()
display(silver_df.limit(20))

root
 |-- subject_id: long (nullable = true)
 |-- hadm_id: long (nullable = true)
 |-- chartdate: date (nullable = true)
 |-- hcpcs_cd: string (nullable = true)
 |-- seq_num: integer (nullable = true)
 |-- short_description: string (nullable = true)



subject_id,hadm_id,chartdate,hcpcs_cd,seq_num,short_description
10000068,25022803,2160-03-04,99218,1,Hospital observation services
10000084,29888819,2160-12-28,G0378,1,Hospital observation per hr
10000108,27250926,2163-09-27,99219,1,Hospital observation services
10000117,22927623,2181-11-15,43239,1,Digestive system
10000117,22927623,2181-11-15,G0378,2,Hospital observation per hr
10000161,22148160,2163-08-20,99219,1,Hospital observation services
10000248,20600184,2192-11-30,99219,1,Hospital observation services
10000280,25852320,2151-03-18,99219,1,Hospital observation services
10000635,20642640,2143-12-23,G0378,1,Hospital observation per hr
10000635,26134563,2136-06-19,G0378,1,Hospital observation per hr


In [0]:
# Report exact duplicate HCPCS event records without removing them or blocking publication
# One aggregation pass over a single groupBy (no exceptAll); duplicates are preserved.
duplicate_summary = (
    silver_df
    .groupBy(*silver_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("surplus_rows"),
    )
    .first()
)

duplicate_count = duplicate_summary["duplicate_groups"]
duplicate_surplus_rows = duplicate_summary["surplus_rows"]

print("Exact duplicate HCPCS event groups [preserved, report only]:", duplicate_count)
print("Rows beyond the first in those groups [preserved, report only]:", duplicate_surplus_rows)

Exact duplicate HCPCS event groups [preserved, report only]: 0
Rows beyond the first in those groups [preserved, report only]: 0


In [0]:
# Check missing event identifiers, codes and optional fields
quality_df = silver_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in silver_df.columns
    ],
    F.sum(
        F.when(F.col("hcpcs_cd") == "", 1).otherwise(0)
    ).alias("blank_hcpcs_codes"),
    F.sum(
        F.when(F.col("short_description") == "", 1).otherwise(0)
    ).alias("blank_descriptions")
)

display(quality_df)

null_subject_id,null_hadm_id,null_chartdate,null_hcpcs_cd,null_seq_num,null_short_description,blank_hcpcs_codes,blank_descriptions
0,0,0,0,0,0,0,0


In [0]:
# Check which event codes match the HCPCS reference dictionary
# Informational only: the reference table is published by 06_d_hcpcs
if spark.catalog.tableExists(f"{SILVER}.d_hcpcs"):
    hcpcs_reference_df = (
        spark.table(f"{SILVER}.d_hcpcs")
        .select(F.col("code").alias("reference_code"))
        .distinct()
    )

    code_coverage_df = (
        silver_df
        .select("hcpcs_cd")
        .distinct()
        .join(
            hcpcs_reference_df,
            F.col("hcpcs_cd") == F.col("reference_code"),
            "left"
        )
        .select(
            "hcpcs_cd",
            F.col("reference_code").isNotNull().alias("found_in_dictionary")
        )
    )

    display(
        code_coverage_df
        .groupBy("found_in_dictionary")
        .count()
    )
else:
    print(f"{SILVER}.d_hcpcs not found; dictionary coverage profiling skipped")

found_in_dictionary,count
true,2366


In [0]:
# Confirm every Bronze record is preserved by row count
bronze_count = bronze_df.count()
silver_count = silver_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"

Bronze rows: 186074
Silver rows: 186074


In [0]:
# Validate required identifiers, HCPCS codes and row counts
quality = quality_df.first().asDict()

# Documented NOT NULL columns in MIMIC-IV hosp.hcpcsevents
required_checks = [
    "null_subject_id",
    "null_hadm_id",
    "null_seq_num",
    "null_hcpcs_cd",
]

issues = {k: quality[k] for k in required_checks if quality[k] != 0}

assert not issues, f"Documented NOT NULL columns contain NULLs: {issues}"
assert bronze_count == silver_count, "Record counts do not match"

print("Blank hcpcs_cd strings [preserved, review only]:", quality["blank_hcpcs_codes"])
print("All critical Silver quality checks passed.")
print("Exact duplicate groups (preserved, review only):", duplicate_count)

Blank hcpcs_cd strings [preserved, review only]: 0
All critical Silver quality checks passed.
Exact duplicate groups (preserved, review only): 0


In [0]:
# Publish the validated HCPCS events table (CREATE OR REPLACE; downstream Gold depends on it)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)

Published: cdr_mimiciv.silver.hcpcsevents


In [0]:
# Verify published HCPCS event records
published_df = spark.table(TARGET_TABLE)

published_count = published_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Published rows:", published_count)

assert published_count == silver_count, "Published row-count mismatch"
published_schema = {f.name: f.dataType.simpleString() for f in published_df.schema.fields}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from contract: {published_schema}"
)

published_quality = published_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in ["subject_id", "hadm_id", "seq_num", "hcpcs_cd"]
    ]
).first()
assert all(v == 0 for v in published_quality.asDict().values()), f"Published NULLs: {published_quality}"

display(published_df.limit(20))

print("Silver hcpcsevents table verified successfully.")

Bronze rows: 186074
Silver rows: 186074
Published rows: 186074


subject_id,hadm_id,chartdate,hcpcs_cd,seq_num,short_description
10000068,25022803,2160-03-04,99218,1,Hospital observation services
10000084,29888819,2160-12-28,G0378,1,Hospital observation per hr
10000108,27250926,2163-09-27,99219,1,Hospital observation services
10000117,22927623,2181-11-15,43239,1,Digestive system
10000117,22927623,2181-11-15,G0378,2,Hospital observation per hr
10000161,22148160,2163-08-20,99219,1,Hospital observation services
10000248,20600184,2192-11-30,99219,1,Hospital observation services
10000280,25852320,2151-03-18,99219,1,Hospital observation services
10000635,20642640,2143-12-23,G0378,1,Hospital observation per hr
10000635,26134563,2136-06-19,G0378,1,Hospital observation per hr


Silver hcpcsevents table verified successfully.


In [0]:
# Finish successfully only after every assertion above has passed
dbutils.notebook.exit(f"{TABLE}:OK")